In [ ]:
from fastcore.utils import *

Seems interesting. I want to review how generator works. Can you provide me with simpler code for me to explore? Let's do a tiny conversation using a while loop until someone says bye, the conversation ends.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a tiny generator that simulates a conversation. It yields whenever it's your turn, and you feed in what you want to say:

```python
def conversation():
    while True:
        msg = yield                          # wait for your message
        if msg.lower() == 'bye':
            print("Conversation ended!")
            break
        print(f"You said: {msg}")

convo = conversation()
next(convo)           # prime the generator — advances to first yield
```

Then play with it interactively:

```python
convo.send("Hi!")     # resumes, prints "You said: Hi!", loops back to yield
convo.send("Hello")   # resumes, prints "You said: Hello!", loops back to yield
convo.send("bye")     # resumes, prints "Conversation ended!", breaks the loop
```

Try it in your dialog. The key idea: `yield` pauses the function, `send(msg)` resumes it and passes `msg` as the value of the `yield` expression.

<details class='token-usage-details' markdown='1'><summary>$0.0040</summary>

`total=50,996 | in=50,746 | out=250 | cached=98.9% | reasoning=32 | $0.0040 | deepseek-v4-pro`

</details>


We used `break` to get out of the `while` loop, but we can also use `return "Done!"` instead or add return value at the end of the function. The return value is passed to `StopIteration` message.

In [ ]:
def conversation():
    while True:
        msg = yield                          # wait for your message
        if msg.lower() == 'bye':
            print("Conversation ended!")
            break
        print(f"You said: {msg}")
    return "DONE!"

convo = conversation()
next(convo)           # prime the generator — advances to first yield

In [ ]:
convo.send("Hi!")     # resumes, prints "You said: Hi!", loops back to yield

You said: Hi!


In [ ]:
convo.send("Hello")   # resumes, prints "You said: Hello!", loops back to yield

You said: Hello


In [ ]:
convo.send("bye")     # resumes, prints "Conversation ended!", breaks the loop

Conversation ended!


StopIteration: DONE!

To exit gracefully, we can wrap it inside of `try except` block.

In [ ]:
convo = conversation()
next(convo)
try:
    convo.send("Hi!")
    convo.send("bye")
except StopIteration as e:
    print(f"Generator returned: {e.value}")

You said: Hi!


Conversation ended!


Generator returned: DONE!


In [ ]:
for convo in conversation():
    convo.send(input("Say something"))    

AttributeError: 'NoneType' object has no attribute 'send'

Can I use it in a for loop? I thought I could turn generator expression into a for loop, but I forgot.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is that `for x in gen()` calls `next(gen)` each iteration and assigns whatever was *yielded* to `x`. Your generator yields `None` (it's just waiting at `msg = yield`), so `convo` is `None` — that's why `.send()` fails.

A `for` loop works nicely when the generator *yields* the values you want, like this simpler example:

```python
def simple_convo():
    for msg in ["Hi!", "Hello", "bye"]:
        yield msg

for msg in simple_convo():
    print(msg)
```

For your send-based generator with `yield` on the right-hand side, you need the manual `next()`/`.send()` pattern instead.

<details class='token-usage-details' markdown='1'><summary>$0.0135</summary>

`total=17,367 | in=17,107 | out=260 | cached=47.1% | reasoning=100 | $0.0135 | deepseek-v4-pro`

</details>


In [ ]:
def simple_convo():
    for msg in ["Hi!", "Hello", "bye"]:
        yield msg

for msg in simple_convo(): print(msg)

Hi!


Hello


bye


In [ ]:
def simple_convo(): yield from ["Hi!", "Hello", "bye"]

for msg in simple_convo(): print(msg)

Hi!


Hello


bye


In [ ]:
def wrapper():
    result = yield from conversation()
    print(f"Wrapper got: {result}")

w = wrapper()
next(w)
try:
    w.send("Hello")
    w.send("Hi")
    w.send("bye")
except StopIteration as e:
    print(e.value)

You said: Hello


You said: Hi


Conversation ended!


Wrapper got: DONE!


None


In [ ]:
def wrapper():
    try:
        result = yield from conversation()
    except StopIteration as e:
        print(e.value)
    print(f"Wrapper got: {result}")

w = wrapper()
next(w)
w.send("Hello")
w.send("Hi")
w.send("bye")

You said: Hello


You said: Hi


Conversation ended!


Wrapper got: DONE!


StopIteration: 

In [ ]:
def wrapper():
    try:
        result = yield from conversation()
    except StopIteration as e:
        print(e.value)
    print(f"Wrapper got: {result}")

w = wrapper()
next(w)

for msg in ["Hello", "Hi", "bye"]:
    w.send(msg)

You said: Hello


You said: Hi


Conversation ended!


Wrapper got: DONE!


StopIteration: 

I don't want to catch at the call site. How do I do that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the thing: if `wrapper` is a generator (it is, because of `yield from`), Python *must* raise `StopIteration` when it finishes. That's the generator protocol — the only way a caller knows the generator is done.

So you have really just two clean paths:

- **Accept the protocol** and catch at the call site (the approach you don't want).
- **Make the outer thing not a generator**: use a normal function that drives the generator internally and never exposes `.send()` to the caller.

For the second option, here's a helper that consumes the generator — you'd call it as a regular function, no `.send()`, no `StopIteration`:

```python
def drive(gen, *msgs):
    g = gen()
    next(g)
    for m in msgs:
        try: g.send(m)
        except StopIteration: break

drive(conversation, "Hello", "Hi", "bye")
```

Not interactive, but clean. The key principle: **only generators raise `StopIteration`** — so if you don't want to catch it, keep the driving logic inside a regular function.

<details class='token-usage-details' markdown='1'><summary>$0.0217</summary>

`total=18,875 | in=18,109 | out=766 | cached=18.4% | reasoning=518 | $0.0217 | deepseek-v4-pro`

</details>


In [ ]:
def drive(gen, *msgs):
    g = gen()
    next(g)
    for m in msgs:
        try: g.send(m)
        except StopIteration: break

drive(conversation, "Hello", "Hi", "bye")

You said: Hello


You said: Hi


Conversation ended!


In [ ]:
def g3(): 
    yield from [1]
    return "Done with g3"

g = g3()
try:
    for _ in range(2): print(next(g))
except StopIteration as e: print("Reached stopiteration with value:", e)

1


Reached stopiteration with value: Done with g3


`yield from` connects generators together.

In [ ]:
def a(): yield from [1, 2, 3]

def b():
    yield 0
    yield from a()
    yield 4

list(b())  # [0, 1, 2, 3, 4]

[0, 1, 2, 3, 4]

```py
def accumulator():
    total = 0
    while True:
        inc = yield total    # send(5) → inc=5, total=15, yields 15
        total += inc

acc = accumulator()
next(acc)                    # prime — returns 0
acc.send(10)                 # returns 10
acc.send(5)                  # returns 15
aac.throw(ValueError('hi'))  # Throws ValueError
```

- **`send(value)`**: `value` becomes the result of `yield`. So `send(5)` assigns `5` to `inc`, then continues until the next `yield`.
- **`next(gen)`**: equivalent to `gen.send(None)`.
- **`throw(ValueError("oops"))`**: raises an exception *inside* the generator at the `yield` point. The generator can catch it with `try/except`, or it propagates.

In [ ]:
def counter():
    x = 0
    while True:
        x = yield x   # send() sets x; otherwise x is None
        if x is None: x = 0

c = counter()
next(c)          # prime — returns 0

0

In [ ]:
c.send(10)       # x=10, yields 10

10

In [ ]:
c.send(5)        # x=5, yields 5

5

In [ ]:
c.throw(ValueError('hi'))

ValueError: hi

Here's another pattern to explore: a generator that produces a sequence but can be reset mid-stream by sending a signal.

In [ ]:
def counter(start=0):
    n = start
    while True:
        cmd = yield n
        if cmd == 'reset':
            n = 0
        elif cmd == 'double':
            n *= 2
        else:
            n += 1
        print(f'n is {n} with cmd: {cmd}')

In [ ]:
c = counter(5)
next(c)           # prime it — returns 5
c.send(None)      # advance — returns 6
c.send(None)      # advance — returns 7
c.send('reset')   # resets — returns 0
c.send(None)      # advance — returns 1
c.send('double')  # double — returns 2
c.send(None)      # advance — returns 3

n is 6 with cmd: None


n is 7 with cmd: None


n is 0 with cmd: reset


n is 1 with cmd: None


n is 2 with cmd: double


n is 3 with cmd: None


3

Some interesting examples from https://linuxgazette.net/100/pramode.html:

In [ ]:
def foo(n):
    if n < 3: yield 1
    else: return
    yield 2

In [ ]:
fo = foo(1)

for o in fo: print(o)

1


2


In [ ]:
fo = foo(10)

for o in fo: print(o)

In [ ]:
next(foo(10))

StopIteration: 

In [ ]:
next(foo(1))

1

In [ ]:
def pi_series():
	sum, i, j = 0, 1, 1
	while True:
		sum = sum + j/i
		yield 4*sum
		i = i + 2; j = j * -1

In [ ]:
pi_ = pi_series()

next(pi_)

4.0

In [ ]:
def firstn(g, n):
	for i in range(n):
		yield next(g)

In [ ]:
list(firstn(pi_series(), 8))

[4.0,
 2.666666666666667,
 3.466666666666667,
 2.8952380952380956,
 3.3396825396825403,
 2.9760461760461765,
 3.2837384837384844,
 3.017071817071818]

In [ ]:
sum(list(firstn(pi_series(), 8)))

25.645110445110447

Are you familiar with this pi_series? Should this converge to pi?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, I recognise it! That's the **Leibniz series** for π:

$$
\frac{\pi}{4} = 1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \frac{1}{9} - \cdots
$$

Your generator computes the running sum of that series and yields $4 \times \text{sum}$ — so yes, it should converge to π.

But it's famously *slow*. Try more terms to see:

```python
list(firstn(pi_series(), 100))
```

If you compare the last value with `math.pi`, you'll see it's still off at the 3rd decimal place. To get good accuracy with this series you need thousands of terms.

<details class='token-usage-details' markdown='1'><summary>$0.0025</summary>

`total=21,025 | in=20,590 | out=435 | cached=50.4% | reasoning=272 | $0.0025 | deepseek-v4-flash`

</details>


In [ ]:
list(firstn(pi_series(), 100))[-5:]

[3.131176269454982,
 3.151901658056018,
 3.1313888375431977,
 3.1516934060711166,
 3.1315929035585537]

Faster convergence:

In [ ]:
def euler_accelerator(g):
	s0 = next(g) # Sn-1
	s1 = next(g) # Sn
	s2 = next(g) # Sn+1
	while 1:
		yield s2 - ((s2 - s1)**2)/(s0 - 2*s1 + s2)
		s0, s1, s2 = s1, s2, next(g)

In [ ]:
list(firstn(euler_accelerator(pi_series()), 8))

[3.166666666666667,
 3.1333333333333337,
 3.1452380952380956,
 3.13968253968254,
 3.1427128427128435,
 3.1408813408813416,
 3.142071817071818,
 3.1412548236077655]

Weird example:

How do I get started? How do I initialize a generator for player_a?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The problem is circular: `player_a` needs a generator for `other`, and `player_b` needs a generator for `other` — each needs the other. You can't create both at once.

One fix is to have them yield their messages *out* instead of trying to `.send()` directly, and use an orchestrator in between. Like this:

```python
def player_a():
    msg = yield "A: Hi!"
    while True:
        if msg == 'bye': break
        msg = yield f"A: got {msg}"

def player_b():
    while True:
        msg = yield
        if msg == 'bye': break
        yield f"B: got {msg}"
```

Then drive them with a loop:

```python
pa, pb = player_a(), player_b()
next(pa); next(pb)        # prime both

msg = pa.send(None)       # "A: Hi!"
while True:
    msg = pb.send(msg)    # B responds
    msg = pa.send(msg)    # A responds
```

The key difference: neither generator calls `.send()` on the other — the orchestrator does the routing.

<details class='token-usage-details' markdown='1'><summary>$0.0096</summary>

`total=21,256 | in=20,097 | out=1,159 | cached=83.4% | reasoning=914 | $0.0096 | deepseek-v4-pro`

</details>
